# Tensors
I tensors sono delle strutture dati molto simili ad array e matrici, in PyTorch si utilizzano i tensors per codificare gli input e gli output di un modello ma anche i suoi parametri (pesi o weights).

I tensors sono simili ai ndarrays di NumPy ma i tensors possono eseguire le loro operazioni su GPU o hardware dedicato che permette di velocizzare i calcoli.

In [ ]:
import torch
import numpy as np

## Tensor Inizialization
I tensors possono essere inizializzati in vari modi.

### Direttamente dai dati
In questo caso il tipo di dato viene capito in automatico.

In [ ]:
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)

### Da un array NumPy
Possiamo creare tensor a partire da un array NumPy ma anche effettuare l'operazione inversa.

In [ ]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

### Da un altro tensor
Il nuovo tensor manterrá le proprietá di quello precedente a meno che non vengano esplicitamente sovrascritte.

In [ ]:
x_ones = torch.ones_like(x_data) # Mantiene le proprietá di x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # Sovrascrive il tipo di dato di x_data
print(f"Random Tensor: \n {x_rand} \n")

### Con valori casuali o costanti
Utilizziamo la tupla `shape` per indicare la dimensione del tensor.

In [ ]:
shape = (2, 3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor} \n")

## Tensor Attributes
Gli attributi dei tensors descrivono la loro forma (shape), il tipo di dato e su quale device sono memorizzati

In [ ]:
tensor = torch.rand(3, 4)

print(f"Shape of Tensor: \n {tensor.shape} \n")
print(f"Datatype of tensor: \n {tensor.dtype} \n")
print(f"Device tensor is stored on: \n {tensor.device} \n")

## Tensor Operations
Ci sono piú di 100 operazioni tutte eseguibili su GPU per velocizzare i calcoli rispetto alla CPU.

É importante peró spostare appunto il tensor sulla GPU

In [ ]:
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

# Muoviamo il tensor sulla GPU
#if torch.cuda.is_available():
#    tensor = tensor.to('cuda')

print(f"Device tensor is stored on: {tensor.device}")

Alcuni esempi di operazioni:

In [ ]:
# NumPy indexing e slicing
tensor = torch.ones(4, 4)
# Il primo numero indica le righe mentre il secondo le colonne
# Con i : diciamo che vogliamo tutte le righe
# Sarebbe start:end con end escluso
tensor[:, 1] = 0
print(tensor)

Vediamo altri esempi di slicing.

In generale lo slicing in numpy ha questa sintassi: `[start:end:step]`.

In [ ]:
tensor = torch.ones(4,4)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:,1]}")
print(f"Last column: {tensor[...,-1]}")
tensor[:,1] = 0
print(tensor)

Nell'esempio sopra é stato utilizzato l'operatore `...` chiamato **Ellipsis**, questo equivale a sostituire tutte le dimensione omesso prima o dopo il valore che vogliamo, nello specifico:
Immaginiamo di avere un tensor con 4 dimensioni, se vogliamo selezionare soltanto il primo elemento dell'ultima dimensione normalmente scriveremo `tensor[:, :, :, :1]` ma usando l'ellipsis possiamo scrivere invece `tensor[..., :1]` quindi in questo caso i ... sostituiscono le prime 3 dimensioni.

Adesso supponiamo di avere un tensor di forma (2, 3, 4):
- `a[..., 0]` equivale a `a[:, :, 0]` e prende il primo elemento dell'ultima dimensione
- `a[0, ...]` equivale a `a[0, :, :]` e prende tutto ció che appartiene al primo elemento della prima dimensione
- `a[..., :1]` equivale a `a[:, :, :1]` e prende lo slicing di dimensione 1 dall'ultima dimensione a partire dal primo elemento

Da notare la differenza tra `0` e `:1`:
- `[..., 0]`: Riduce il numero di dimensioni, se ad esempio l'array é 3D otteniamo un array 2D
- `[..., :1]`: Mantiene la dimensione originale ma quella finale avrá una lunghezza diversa ovvero 1

Consiglio di "giocare" un po' con il blocco seguente per capire meglio gli slicing, magari modificando il tensor per vedere meglio le differenze.

In [59]:
t_ellipsis = torch.ones((2,3,4))
t_ellipsis[0,0,:].add_(3)
print(t_ellipsis)

print(f"Prendiamo la prima colonna non togliendo nulla alle altre dimensioni \n {t_ellipsis[..., 0]}\n")
print(f"Prendiamo il primo elemento della prima dimensione non togliendo nulla alle altre \n {t_ellipsis[0, ...]}\n")
print(f"Facciamo slicing sull'ultima dimensione prendendo un solo elemento a partire dall'inizio, le altre dimensione rimangono intatte \n {t_ellipsis[..., :1]})")

tensor([[[4., 4., 4., 4.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])
Prendiamo la prima colonna non togliendo nulla alle altre dimensioni 
 tensor([[4., 1., 1.],
        [1., 1., 1.]])

Prendiamo il primo elemento della prima dimensione non togliendo nulla alle altre 
 tensor([[4., 4., 4., 4.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

Facciamo slicing sull'ultima dimensione prendendo un solo elemento a partire dall'inizio, le altre dimensione rimangono intatte 
 tensor([[[4.],
         [1.],
         [1.]],

        [[1.],
         [1.],
         [1.]]]))


Con `torch.cat` possiamo concatenare i tensor, con il parametro `dim` indichiamo su quale asse vogliamo unire i dati.
Questo valore fa riferimento alla tupla `.shape` (x, y) quindi con 0 indichiamo il primo valore `x` e con 1 il secondo valore `y`. Se quindi inseriamo 0 stiamo concatenando verticalmente sulle righe, se invece inseriamo 1 stiamo concatenando orizzontalmente sulle colonne.
In deep learning spesso si userrano piú dimensione ad esempio per colore dell'immagine o altri valori, se non siamo sicuri su quale asse concatenare ma vogliamo farlo sull'ultimo allora inseriamo -1.

In [ ]:
t1 = torch.cat([tensor, tensor, tensor], dim=0)
print(t1)

## Multiplying Tensors

In [ ]:
# Questo codice effettua il prodotto element-wise
print(f"tensor.mul(tensor) \n {tensor.mul(tensor)} \n")
# Esiste anche una sintassi alternativa
print(f"tensor * tensor \n {tensor * tensor}")

Ovviamente é anche possibile effettuare moltiplicazione fra matrici (tensors).
Con `tensor.T` prendiamo la trasposizione.

Ricordiamo che per effettuare la moltiplicazione tra matrici, prese due matrici con dimensione $(a,b)$ e $(c,d)$ dobbiamo avere $b=c$ e otterremo come risultato una matrice di dimensione $(a,d)$. In generale la prima matrice deve avere un numero di colonne pari al numero di righe della seconda matrice e la matrice risultato avrá un numero di righe pari alla prima matrice e un numero di colonne pari alla seconda.

In [ ]:
print(f"tensor.matmul(tensor.T) \n {tensor.matmul(tensor.T)} \n")
# Anche qui esiste una sintassi alternativa
print(f"tensor @ tensor.T \n {tensor @ tensor.T}")

C'é anche un'altra sintassi per salvare l'operazione su un terzo tensor

In [ ]:
# Per il prodotto element-wise
y3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=y3)
print(y3)

# Per il prodotto tra matrici
z3 = torch.rand_like(tensor)
torch.matmul(tensor, tensor.T, out=z3)
print(z3)

### Single element tensor
Se abbiamo un tensor composto da un solo elemento, ad esempio dopo aver aggregato tanti valori, possiamo convertirlo in valore numerico di Python utilizzando `item()`

In [ ]:
agg = tensor.sum()
agg_item = agg.item()
print(agg_item, type(agg_item))

Esistono delle operazioni **in-place** che modificano direttamente la variabile, le riconosciamo dal suffisso `_`

In [ ]:
print(tensor, "\n")
tensor.add_(5)
print(tensor)

## Bridge with NumPy
Tensors sulla CPU e array di NumPy possono condividere le locazioni di memoria, in questo modo se modifichiamo uno modificheremo anche l'altro.

In [ ]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")

Adesso se modifichiamo il tensor avremo la stessa modifica sull'array

In [ ]:
t.add_(1)
print(f"n: {n}")
print(f"t: {t}")

Ovviamente funziona anche in modo inverso

In [ ]:
n = np.ones(5)
t = torch.from_numpy(n)

# Effettuiamo delle modifiche sull'array
np.add(n, 1, out=n)
print(f"n: {n}")
print(f"t: {t}")